# deficit_pct_gdp.ipynb

Fetches FRED series **FYFSGDA188S** (Federal Surplus or Deficit as % of GDP, annual)
and writes `viz/public/deficit_pct_gdp.csv`.

**Source:** https://fred.stlouisfed.org/series/FYFSGDA188S  
**Coverage:** 1929–present  
**Units:** Percent of GDP (negative = deficit, positive = surplus)

**Output schema:**
```
year,deficit_pct_gdp
1929,-0.643
1930,0.773
...
```

No API key required — FRED public data endpoint.

In [2]:
import pandas as pd
from pathlib import Path
import urllib.request

FRED_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=FYFSGDA188S"
OUT_FILE = Path("../public/deficit_pct_gdp.csv")

print(f"Fetching: {FRED_URL}")
print(f"Output:   {OUT_FILE.resolve()}")

Fetching: https://fred.stlouisfed.org/graph/fredgraph.csv?id=FYFSGDA188S
Output:   /Users/aaditbhatia/Desktop/OBBB Dashboard/viz/public/deficit_pct_gdp.csv


In [3]:
# ── Fetch from FRED ────────────────────────────────────────────────────────

with urllib.request.urlopen(FRED_URL) as response:
    raw = pd.read_csv(response)

print(f"Rows fetched: {len(raw)}")
print(raw.head())

URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)>

In [ ]:
# ── Clean and reshape ──────────────────────────────────────────────────────
# FRED returns DATE (YYYY-01-01) and FYFSGDA188S (float or '.' for missing)

df = raw.copy()
df.columns = ["date", "deficit_pct_gdp"]

# Parse year from date string
df["year"] = pd.to_datetime(df["date"]).dt.year

# Drop missing values (FRED uses '.' for unreleased observations)
df["deficit_pct_gdp"] = pd.to_numeric(df["deficit_pct_gdp"], errors="coerce")
df = df.dropna(subset=["deficit_pct_gdp"])

# Keep only year and value, sorted
df = df[["year", "deficit_pct_gdp"]].sort_values("year").reset_index(drop=True)

print(f"Coverage: {df['year'].min()} – {df['year'].max()}  ({len(df)} years)")
print(df.tail(8).to_string(index=False))

In [ ]:
# ── Sanity checks ──────────────────────────────────────────────────────────
# Known approximate values from FRED:

checks = {
    1944: (-25, -18),   # WWII peak
    1969: (-0.5, 1.0),  # last pre-Clinton surplus
    1983: (-7,   -5),   # Reagan
    2000: ( 1.5,  3.0), # Clinton surplus peak
    2009: (-11,  -8),   # GFC
    2020: (-17, -12),   # COVID
    2024: (-7.5, -5),   # recent
}

all_ok = True
for year, (lo, hi) in checks.items():
    row = df[df["year"] == year]
    if row.empty:
        print(f"  MISSING  {year}")
        all_ok = False
        continue
    val = row["deficit_pct_gdp"].iloc[0]
    ok  = lo <= val <= hi
    print(f"  {'OK  ' if ok else 'FAIL'}  {year}: {val:+.2f}%  (expected {lo} to {hi})")
    if not ok:
        all_ok = False

print()
print("All checks passed ✓" if all_ok else "Some checks failed — do not write until resolved.")

In [ ]:
# ── Write CSV ──────────────────────────────────────────────────────────────
# Only run once all sanity checks above pass.

OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_FILE, index=False)

print(f"Written {len(df)} rows → {OUT_FILE}")
print(df.tail(6).to_string(index=False))